In [3]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta

# 获取当前日期和时间
now = datetime.now()
today_time = now.strftime("%Y-%m-%d 00:00:00")
today_time = datetime.strptime(today_time, "%Y-%m-%d %H:%M:%S")

query = f"""name:点击立即定制 |select phone,uid,count(0)cnt from log group by 1,2"""

click_uid_df = get_sls_data_by_query(
    query=query,
    from_time=today_time,
    to_time=now,
    project="xianmu-front-end-log",
    logstore="xm-mall",
)

click_uid_df=click_uid_df.drop(columns=["__source__", "__time__"], errors="ignore")
click_uid_df.head(4)

即将获取数据: =====> 2025-04-07 00:00:00 2025-04-07 18:46:46.861333 xm-mall: name:点击立即定制 |select phone,uid,count(0)cnt from log group by 1,2
>=====数条数:14


,phone,uid,cnt
0,18168981994,472735,1
1,13817287516,51637,1
2,18601684860,496243,1
3,18252612803,252894,1


In [4]:
# 获取token
import requests
import os

login_url = "https://admin.summerfarm.net/authentication/auth/username/login"
login_data = {
    "username": "peng.tang@summerfarm.net",
    "password": os.getenv("XIANMU_ADMIN_PASSWORD"),
}

token = requests.post(login_url, data=login_data).json()

token_str = token.get("data").get("token")

print(token)


headers = {
    "token": token_str,
    "xm-rqid": "create_fake_merchant_tp",
    "xm-uid": "2047",
    "Content-Type": "application/json;charset=UTF-8",
}

print(headers)

{'code': 'SUCCESS', 'data': {'authId': 2082, 'bizUserId': 2047, 'phone': '18618107293', 'realname': '唐鹏', 'token': 'admin__690183e7-5736-4f8a-9c22-ff4a615af177', 'userBaseId': 2080}, 'msg': '', 'success': True}
{'token': 'admin__690183e7-5736-4f8a-9c22-ff4a615af177', 'xm-rqid': 'create_fake_merchant_tp', 'xm-uid': '2047', 'Content-Type': 'application/json;charset=UTF-8'}


In [11]:
import requests


def get_merchant_detail(uid=51637):
    # curl 'https://admin.summerfarm.net/sf-mall-manage/merchant/query/page' \
    #   -H 'content-type: application/json;charset=UTF-8' \
    #   -H 'token: admin__65c97b2e-1e21-4d2b-a3c4-dbedd161d0f7' \
    #   --data-raw '{"mId":51637,"pageIndex":1,"pageSize":10}'

    uid = int(uid)

    url = "https://admin.summerfarm.net/sf-mall-manage/merchant/query/page"
    _headers = {
        "content-type": "application/json;charset=UTF-8",
        "token": headers["token"],
    }
    data = {"mId": uid, "pageIndex": 1, "pageSize": 10}
    response = requests.post(url, headers=_headers, json=data)
    return response.json().get("data", {}).get("list", [])[0]


print(get_merchant_detail())

click_uid_df["merchant"] = click_uid_df["uid"].apply(get_merchant_detail)
click_uid_df.head(4)

{'adminRealName': '董元鸣', 'area': '普陀区', 'areaName': '上海', 'areaNo': 2750, 'auditTime': '2020-12-15 16:35:04', 'businessType': '茶饮', 'channelCode': '2kBkAD', 'city': '上海市', 'companyBrand': '上海路飞餐饮管理有限公司', 'grade': 0, 'mId': 51637, 'mcontact': 'Private', 'mname': '路飞堂泸定路店', 'operateStatus': 0, 'phone': '13817287516', 'poiNote': '121.388108,31.227402', 'popMerchant': False, 'province': '上海', 'registerTime': '2020-12-15 16:35:02', 'size': '单店', 'status': 1, 'storeId': 12988, 'type': '未知', 'updateTime': '2023-09-04 17:23:23'}
